In [ ]:
import requests
import time
import pandas as pd
from sqlalchemy import create_engine
import mysql.connector
import datetime

# Токен для авторизации
token = '**************************************'

headers = {
    'Authorization': f'OAuth {token}',  # Используем 'OAuth'
    'Content-Type': 'application/json',
}

In [ ]:
current_date = datetime.date.today()
before_yesterday_date = current_date - datetime.timedelta(days=3)

In [ ]:
data = {
    "log_request": {
        "counter_id": 90602537,
        "source": "hits",
        "date1": str(before_yesterday_date),
        "date2": str(before_yesterday_date),
        "fields": [
            "ym:pv:link",
            "ym:pv:URL",
            "ym:pv:date"
        ],
        "status": "created",
        "size": 0,
        "attribution": "LASTSIGN"
    }
}

url = f'https://api-metrika.yandex.net/management/v1/counter/{data["log_request"]["counter_id"]}/logrequests?date1={data["log_request"]["date1"]}&date2={data["log_request"]["date2"]}&fields={",".join(data["log_request"]["fields"])}&source={data["log_request"]["source"]}'
response = requests.post(url, headers=headers, json=data)

In [ ]:
try:
    request_id = response.json()['log_request']['request_id']
except KeyError:
    print("Error: 'log_request' key not found in response.")
    print(response.text)
    exit()

max_retries = 10
retry_interval = 15

In [ ]:
for attempt in range(1, max_retries + 1):
    time.sleep(retry_interval)
    url_log_request = f'https://api-metrika.yandex.ru/management/v1/counter/90602537/logrequest/{request_id}'
    response = requests.get(url_log_request, headers=headers)

    if not response.content:
        print(f"Attempt {attempt}, Empty content")
        continue

    try:
        status = response.json().get('log_request', {}).get('status')
    except requests.exceptions.JSONDecodeError:
        print(f"Attempt {attempt}, Non-JSON content: {response.text}")
        continue

    print(f"Attempt {attempt}, Status: {status}")

    if status == 'processed':
        url_download = f'https://api-metrika.yandex.ru/management/v1/counter/90602537/logrequest/{request_id}/part/0/download'
        response = requests.get(url_download, headers=headers)

        if not response.content:
            print("Empty content in download response")
        else:
            lines = [line.split('\t') for line in response.text.strip().split('\n')]
            df = pd.DataFrame(lines, columns=['views', 'url', 'date'])
            df = df.iloc[1:]
            df['views'] = pd.to_numeric(df['views'], errors='coerce')
            df = df[df['views'] > 0]

            username = 'report_bd'
            password = 'iB6mR3nM2m'
            database_name = 'report_bd'
            table_name = 'yandex_metrika_external'
            host = '5.35.85.218'

            engine = create_engine(f'mysql+mysqlconnector://{username}:{password}@{host}/{database_name}', echo=False)

            try:
                df.to_sql(name=table_name, con=engine, if_exists='append', index=False)
                print("Data successfully inserted into the database.")
            except Exception as e:
                print(f"Error inserting data into the database: {str(e)}")
        break